In [1]:
import numpy as np
import pandas as pd, numpy as np
import os, sys

def walk_forward_splits(n, min_train=1008, test_size=252, embargo=21):
    # expanding-window walk-forward; embargo = purge gap dropped at each train->test boundary
    # train = [0, test_start - embargo), test = [test_start, test_start + test_size)
    # last train label reaches index (test_start-embargo-1)+h; with embargo=h it lands exactly at test_start-1 -> no leak
    test_start = min_train + embargo
    while test_start + test_size <= n:
        train_idx = np.arange(0, test_start - embargo)
        test_idx  = np.arange(test_start, test_start + test_size)
        yield train_idx, test_idx
        test_start += test_size

In [2]:
from google.colab import drive
drive.mount("/content/drive")

PROC = "/content/drive/MyDrive/volatility-forecast/data/processed"  # drive mounted
df = pd.read_csv(f"{PROC}/dataset.csv", index_col=0, parse_dates=True)

feat = ['qqq_ret','hyg_ret','lqd_ret','tlt_ret','gld_ret','vix_lvl','vix_chg',
        'tnx_lvl','tnx_chg','irx_lvl','irx_chg','slope_lvl','slope_chg',
        'credit_lvl','credit_chg','rv1','rv5','rv21']
tgt = ['y_rv1','y_rv5','y_rv21']

work = df.dropna(subset=feat+tgt)   # contiguous middle; shared frame across all h (y_rv21 binds)
print("full:", len(df), "working:", len(work),
      "| head trim:", df.index.get_loc(work.index[0]),
      "| tail trim:", len(df)-1-df.index.get_loc(work.index[-1]))

folds = list(walk_forward_splits(len(work)))
print("folds:", len(folds))
for k,(tr,te) in enumerate(folds):
    gap = te[0]-tr[-1]-1
    print(f"f{k:02d} train[0:{len(tr):4d}] {work.index[0].date()}->{work.index[tr[-1]].date()}"
          f" | gap {gap} | test {work.index[te[0]].date()}->{work.index[te[-1]].date()} n={len(te)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
full: 3873 working: 3831 | head trim: 21 | tail trim: 21
folds: 11
f00 train[0:1008] 2011-02-02->2015-02-04 | gap 21 | test 2015-03-09->2016-03-07 n=252
f01 train[0:1260] 2011-02-02->2016-02-04 | gap 21 | test 2016-03-08->2017-03-07 n=252
f02 train[0:1512] 2011-02-02->2017-02-03 | gap 21 | test 2017-03-08->2018-03-07 n=252
f03 train[0:1764] 2011-02-02->2018-02-05 | gap 21 | test 2018-03-08->2019-03-08 n=252
f04 train[0:2016] 2011-02-02->2019-02-06 | gap 21 | test 2019-03-11->2020-03-09 n=252
f05 train[0:2268] 2011-02-02->2020-02-06 | gap 21 | test 2020-03-10->2021-03-09 n=252
f06 train[0:2520] 2011-02-02->2021-02-05 | gap 21 | test 2021-03-10->2022-03-08 n=252
f07 train[0:2772] 2011-02-02->2022-02-04 | gap 21 | test 2022-03-09->2023-03-09 n=252
f08 train[0:3024] 2011-02-02->2023-02-07 | gap 21 | test 2023-03-10->2024-03-11 n=252
f09 train[0:3276] 2011-02-02->

In [3]:
# persistence: forward-h vol predicted by trailing-h vol (same column family)
pairs = {1:('rv1','y_rv1'), 5:('rv5','y_rv5'), 21:('rv21','y_rv21')}

rows = []
for h,(xcol,ycol) in pairs.items():
    per_fold = []
    for k,(tr,te) in enumerate(folds):
        yhat = work[xcol].values[te]   # prediction = trailing rv_h at t
        ytru = work[ycol].values[te]   # target = forward rv_h
        rmse = np.sqrt(np.mean((yhat-ytru)**2))
        per_fold.append(rmse)
    per_fold = np.array(per_fold)
    rows.append({'h':h,'rmse_mean':per_fold.mean(),'rmse_std':per_fold.std(),
                 'rmse_min':per_fold.min(),'rmse_max':per_fold.max()})
    print(f"h={h:2d} | per-fold RMSE: "+" ".join(f"{v:.4f}" for v in per_fold))

base = pd.DataFrame(rows).set_index('h')
print()
print(base.round(4))

h= 1 | per-fold RMSE: 0.0113 0.0066 0.0077 0.0126 0.0107 0.0177 0.0113 0.0169 0.0095 0.0113 0.0131
h= 5 | per-fold RMSE: 0.0063 0.0045 0.0050 0.0065 0.0067 0.0104 0.0044 0.0079 0.0044 0.0047 0.0091
h=21 | per-fold RMSE: 0.0053 0.0037 0.0039 0.0060 0.0116 0.0103 0.0040 0.0046 0.0021 0.0047 0.0080

    rmse_mean  rmse_std  rmse_min  rmse_max
h                                          
1      0.0117    0.0032    0.0066    0.0177
5      0.0064    0.0019    0.0044    0.0104
21     0.0058    0.0028    0.0021    0.0116


In [4]:
%cd /content/drive/MyDrive/volatility-forecast


if "" not in sys.path:
    sys.path.insert(0, "")

/content/drive/MyDrive/volatility-forecast


In [5]:
from src.splits import walk_forward_splits
from src.metrics import rmse
from src.data import load_work_frame

work = load_work_frame("data/processed/dataset.csv")
print("work rows:", len(work))          # expect 3831

H_MAX = 21
folds = list(walk_forward_splits(len(work)))
print("n folds:", len(folds))           # expect 11

for k, (tr, te) in enumerate(folds):
    last_train, first_test = tr[-1], te[0]
    assert tr[0] == 0,                        f"f{k}: train does not start from 0"
    assert first_test - last_train - 1 == 21, f"f{k}: gap != embargo"
    assert last_train + H_MAX < first_test,   f"f{k}: label leaked to test"
print("all fold: expanding from 0, gap=21, label leak 0 ✓")

for k, (tr, te) in enumerate(folds):
    print(f"f{k:02d}  train[0:{tr[-1]+1}]  test[{te[0]}:{te[-1]+1}]")

work rows: 3831
n folds: 11
all fold: expanding from 0, gap=21, label leak 0 ✓
f00  train[0:987]  test[1008:1260]
f01  train[0:1239]  test[1260:1512]
f02  train[0:1491]  test[1512:1764]
f03  train[0:1743]  test[1764:2016]
f04  train[0:1995]  test[2016:2268]
f05  train[0:2247]  test[2268:2520]
f06  train[0:2499]  test[2520:2772]
f07  train[0:2751]  test[2772:3024]
f08  train[0:3003]  test[3024:3276]
f09  train[0:3255]  test[3276:3528]
f10  train[0:3507]  test[3528:3780]


In [6]:
for m in list(sys.modules):          # clear cache so fresh src.har loads
    if m.startswith("src"):
        del sys.modules[m]

from src.har import fit_har, predict_har

FEATURES = ["rv1", "rv5", "rv21"]
HORIZONS = {"h1": "y_rv1", "h5": "y_rv5", "h21": "y_rv21"}

folds = list(walk_forward_splits(len(work)))
har_results = {h: [] for h in HORIZONS}   # per-horizon list of per-fold RMSE

for h_name, y_col in HORIZONS.items():
    for tr, te in folds:
        X_tr = work[FEATURES].iloc[tr].values
        y_tr = work[y_col].iloc[tr].values
        X_te = work[FEATURES].iloc[te].values
        y_te = work[y_col].iloc[te].values

        beta = fit_har(X_tr, y_tr)
        yhat = predict_har(beta, X_te)
        har_results[h_name].append(rmse(y_te, yhat))

for h_name in HORIZONS:
    arr = np.array(har_results[h_name])
    print(f"HAR {h_name:3s}  mean {arr.mean():.4f}  std {arr.std():.4f}  "
          f"min {arr.min():.4f}  max {arr.max():.4f}")

HAR h1   mean 0.0087  std 0.0028  min 0.0055  max 0.0148
HAR h5   mean 0.0055  std 0.0022  min 0.0035  max 0.0111
HAR h21  mean 0.0049  std 0.0026  min 0.0020  max 0.0122


In [7]:
H_COL = "y_rv21"          # focus on h=21, where the crisis story is sharpest
FEATURES = ["rv1", "rv5", "rv21"]
PERSIST_COL = "rv21"      # persistence: copy trailing rv21 as the forecast

folds = list(walk_forward_splits(len(work)))

print(f"{'fold':>4} {'persist':>9} {'HAR':>9} {'HAR win':>9}")
for k, (tr, te) in enumerate(folds):
    y_te = work[H_COL].iloc[te].values

    # persistence: yhat = trailing rv21 (no fit)
    yhat_p = work[PERSIST_COL].iloc[te].values
    rmse_p = rmse(y_te, yhat_p)

    # HAR: fit on train, predict on test
    X_tr = work[FEATURES].iloc[tr].values
    y_tr = work[H_COL].iloc[tr].values
    X_te = work[FEATURES].iloc[te].values
    beta = fit_har(X_tr, y_tr)
    yhat_h = predict_har(beta, X_te)
    rmse_h = rmse(y_te, yhat_h)

    win = (rmse_p - rmse_h) / rmse_p * 100   # positive = HAR better
    flag = "" if win > 0 else "  <-- LOST"
    print(f"f{k:02d}  {rmse_p:9.4f} {rmse_h:9.4f} {win:8.1f}%{flag}")

fold   persist       HAR   HAR win
f00     0.0050    0.0043     14.6%
f01     0.0041    0.0033     19.1%
f02     0.0036    0.0034      7.4%
f03     0.0061    0.0052     14.3%
f04     0.0050    0.0040     19.8%
f05     0.0146    0.0122     16.7%
f06     0.0043    0.0040      6.8%
f07     0.0045    0.0056    -25.1%  <-- LOST
f08     0.0022    0.0020     11.9%
f09     0.0038    0.0031     18.4%
f10     0.0084    0.0068     18.9%


In [8]:
# finding out which fold maps to specific dates
folds = list(walk_forward_splits(len(work)))

print(f"{'fold':>4}  {'test start':>12}  {'test end':>12}  {'rows':>5}")
for k, (tr, te) in enumerate(folds):
    start = work.index[te[0]]
    end   = work.index[te[-1]]
    print(f"f{k:02d}  {str(start.date()):>12}  {str(end.date()):>12}  {len(te):>5}")

fold    test start      test end   rows
f00    2015-02-05    2016-02-04    252
f01    2016-02-05    2017-02-03    252
f02    2017-02-06    2018-02-05    252
f03    2018-02-06    2019-02-06    252
f04    2019-02-07    2020-02-06    252
f05    2020-02-07    2021-02-05    252
f06    2021-02-08    2022-02-04    252
f07    2022-02-07    2023-02-07    252
f08    2023-02-08    2024-02-08    252
f09    2024-02-09    2025-02-11    252
f10    2025-02-12    2026-02-12    252
